# 1. Define Bangalore Zones

We define 16 major Bangalore zones with:

- Zone ID
- Zone Name
- Latitude & Longitude

These coordinates will be used to compute real road distances.

In [1]:
# Fetch REAL road distances between Bangalore zones using the FREE
# OpenStreetMap OSRM API.

import requests
import pandas as pd
import numpy as np

zones = {
    0: {
        "name": "Whitefield",
        "lat": 12.9698,
        "lon": 77.7500
    },
    1: {
        "name": "Koramangala",
        "lat": 12.9352,
        "lon": 77.6245
    },
    2: {
        "name": "Indiranagar",
        "lat": 12.9784,
        "lon": 77.6408
    },
    3: {
        "name": "Hebbal",
        "lat": 13.0352,
        "lon": 77.5970
    },
    4: {
        "name": "Marathahalli",
        "lat": 12.9591,
        "lon": 77.6972
    },
    5: {
        "name": "Electronic City",
        "lat": 12.8399,
        "lon": 77.6770
    },
    6: {
        "name": "Jayanagar",
        "lat": 12.9308,
        "lon": 77.5838
    },
    7: {
        "name": "Rajajinagar",
        "lat": 12.9856,
        "lon": 77.5533
    },
    8: {
        "name": "Yeshwanthpur",
        "lat": 13.0237,
        "lon": 77.5406
    },
    9: {
        "name": "BTM Layout",
        "lat": 12.9166,
        "lon": 77.6101
    },
    10: {
        "name": "HSR Layout",
        "lat": 12.9116,
        "lon": 77.6389
    },
    11: {
        "name": "Bannerghatta Rd",
        "lat": 12.8933,
        "lon": 77.5969
    },
    12: {
        "name": "Yelahanka",
        "lat": 13.1007,
        "lon": 77.5963
    },
    13: {
        "name": "Sarjapur Road",
        "lat": 12.9010,
        "lon": 77.6810
    },
    14: {
        "name": "MG Road",
        "lat": 12.9756,
        "lon": 77.6086
    },
    15: {
        "name": "Banashankari",
        "lat": 12.9255,
        "lon": 77.5468
    }
}

# 2. Extract Zone Identifiers

We create:

- `ids` → list of zone indices
- `names` → list of zone names

These help in labeling the distance matrix.

In [2]:
ids = list(zones.keys())
names = [zones[i]["name"] for i in ids]

# 3. Build Coordinate String

We construct a coordinate string in the format:

`lon,lat;lon,lat;...`

This format is required by the OSRM API for multi-point routing queries.

In [3]:
# Build coordinate string: lon,lat;lon,lat;...
coords_str = ";".join([f"{zones[i]['lon']},{zones[i]['lat']}" for i in ids])
coords_str

'77.75,12.9698;77.6245,12.9352;77.6408,12.9784;77.597,13.0352;77.6972,12.9591;77.677,12.8399;77.5838,12.9308;77.5533,12.9856;77.5406,13.0237;77.6101,12.9166;77.6389,12.9116;77.5969,12.8933;77.5963,13.1007;77.681,12.901;77.6086,12.9756;77.5468,12.9255'

# 4. Fetch and Process Distance Matrix

We use the OpenStreetMap OSRM API to compute real-world road distances.

**Process:**

1. Create API request using `/table/v1/driving/`
2. Send GET request to OSRM server
3. Retrieve response in JSON format
4. Extract distance matrix (in meters)
5. Convert meters → kilometers
6. Round values to 2 decimal places

**Output:**

- A NumPy distance matrix (in km) ready for analysis

In [4]:
url = (
    f"http://router.project-osrm.org/table/v1/driving/{coords_str}?annotations=distance"
)

response = requests.get(url, timeout=60)

if response.status_code == 200:
    data = response.json()

    # Distance matrix: meters → kilometers
    dist_km = np.round(np.array(data["distances"]) / 1000, 2)

    # Save distance matrix
    df_dist = pd.DataFrame(dist_km, index=names, columns=names)
    df_dist.index.name = "Zone"
    df_dist.to_csv("distance_matrix_osrm.csv")

    print("\nSample distances (km): top-left 5x5:")
    print(df_dist.iloc[:5, :5].to_string())

else:
    print(f"Error {response.status_code}: {response.text}")
    print("Try again.")


Sample distances (km): top-left 5x5:
              Whitefield  Koramangala  Indiranagar  Hebbal  Marathahalli
Zone                                                                    
Whitefield          0.00        16.94        14.61   25.97          7.30
Koramangala        17.25         0.00         5.99   15.35         10.77
Indiranagar        14.58         6.00         0.00   11.65          8.10
Hebbal             23.13        15.69        11.32    0.00         19.38
Marathahalli        7.02        11.19         8.86   20.22          0.00
